# Customer Support Tone - Local Mistral Zero-Shot Regression

This notebook uses a local Mistral model through Ollama. It trains on no labels, evaluates zero-shot predictions on the supplied `validation.csv`, reports MAE, and creates a test submission.

In [1]:
from pathlib import Path
import json
import re
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = DATA_DIR / 'customersupporttone'

# Experiment settings: change these values and rerun from this cell onward.
OLLAMA_URL = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'mistral'
TEMPERATURE = 0.0
NUM_PREDICT = 3
REQUEST_TIMEOUT_SECONDS = 120

train_df = pd.read_csv(DATA_DIR / 'train.csv')
dev_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Train rows: {len(train_df)} | validation rows: {len(dev_df)} | test rows: {len(test_df)}')

Train rows: 302 | validation rows: 103 | test rows: 101


## Mistral zero-shot prompt

The prompt contains no labeled examples. Mistral must return one integer from 1 to 5, where 1 is the least negative/calm tone and 5 is the most negative/escalated tone.

In [2]:
SYSTEM_PROMPT = 'Classify customer support tone severity. Return exactly one integer from 1 to 5 and nothing else. 1 means calm/positive, 3 means neutral or mixed, and 5 means highly negative, angry, or escalated. Judge the customer messages, especially the last customer message.'

def build_prompt(conversation, last_customer_message):
    return f'{SYSTEM_PROMPT}\n\nConversation:\n{str(conversation).strip()}\n\nLast customer message:\n{str(last_customer_message).strip()}\n\nTone label (1-5):'

print(build_prompt('customer: My order is late | agent: We are sorry.', 'My order is still late.'))

Classify customer support tone severity. Return exactly one integer from 1 to 5 and nothing else. 1 means calm/positive, 3 means neutral or mixed, and 5 means highly negative, angry, or escalated. Judge the customer messages, especially the last customer message.

Conversation:
customer: My order is late | agent: We are sorry.

Last customer message:
My order is still late.

Tone label (1-5):


In [3]:
def parse_label(response_text):
    match = re.fullmatch(r'\s*\(?([1-5])\)?(?:\s*[().,:-])?\s*', response_text)
    if match is None:
        raise ValueError(f'Mistral returned a non-label response: {response_text!r}')
    return int(match.group(1))


def mistral_score(conversation, last_customer_message):
    payload = {
        'model': OLLAMA_MODEL,
        'prompt': build_prompt(conversation, last_customer_message),
        'stream': False,
        'options': {'temperature': TEMPERATURE, 'num_predict': NUM_PREDICT},
    }
    request = Request(OLLAMA_URL, data=json.dumps(payload).encode('utf-8'), headers={'Content-Type': 'application/json'}, method='POST')
    try:
        with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            result = json.loads(response.read().decode('utf-8'))
    except HTTPError as exc:
        body = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'Ollama HTTP {exc.code}: {body}') from exc
    except URLError as exc:
        raise RuntimeError('Could not connect to Ollama. Start Ollama and run: ollama pull mistral') from exc
    if 'response' not in result:
        raise RuntimeError(f'Ollama response did not contain text: {result}')
    return parse_label(result['response'])


def predict_frame(frame):
    predictions = []
    for row_number, row in enumerate(frame.itertuples(index=False), start=1):
        predictions.append(mistral_score(row.conversation, row.last_customer_message))
        if row_number % 10 == 0 or row_number == len(frame):
            print(f'Predicted {row_number}/{len(frame)} rows')
    return np.asarray(predictions, dtype=float)

print(f'Configured local model: {OLLAMA_MODEL}')

Configured local model: mistral


## Validation MAE

In [4]:
dev_predictions = predict_frame(dev_df)
dev_actual = dev_df['label'].to_numpy(dtype=float)
validation_mae = mean_absolute_error(dev_actual, dev_predictions)
print(f'Validation MAE: {validation_mae:.4f}')

Predicted 10/103 rows
Predicted 20/103 rows
Predicted 30/103 rows
Predicted 40/103 rows
Predicted 50/103 rows
Predicted 60/103 rows
Predicted 70/103 rows
Predicted 80/103 rows
Predicted 90/103 rows
Predicted 100/103 rows
Predicted 103/103 rows
Validation MAE: 1.0000


In [5]:
from sklearn.metrics import f1_score

dev_macro_f1 = f1_score(dev_actual.astype(int), dev_predictions.astype(int), average='macro')
print(f'Development macro-F1: {dev_macro_f1:.4f}')

Development macro-F1: 0.2730


## Test submission

In [6]:
test_predictions = predict_frame(test_df)
submission = pd.DataFrame({'id': test_df['id'], 'label': test_predictions})
submission_path = DATA_DIR / 'submission_mistral_zero_shot.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to {submission_path}')
submission.head()

Predicted 10/101 rows
Predicted 20/101 rows
Predicted 30/101 rows
Predicted 40/101 rows
Predicted 50/101 rows
Predicted 60/101 rows
Predicted 70/101 rows
Predicted 80/101 rows
Predicted 90/101 rows
Predicted 100/101 rows
Predicted 101/101 rows
Saved 101 predictions to c:\Users\bradm\Documents\GitHub\ai-masters-workspace\ARI-510-Machine-Learning\Phase 2b\customersupporttone\submission_mistral_zero_shot.csv


,id,label
0,18972,3.0
1,49147,5.0
2,86244,5.0
3,98577,5.0
4,99913,5.0


In [ ]:
if 'label' in test_df.columns:
    test_macro_f1 = f1_score(test_df['label'], test_predictions.astype(int), average='macro')
    print(f'Test macro-F1: {test_macro_f1:.4f}')
else:
    print('Test macro-F1 unavailable: test.csv has no ground-truth label column.')